#### Imports

In [37]:
from student.agent import *
from student.agent.memory_rag import MemoryRAG, MemoryNodeRAG
import pandas as pd
import json
from student.agent.agent_baselines import BaselineAgent

### Config

In [38]:
training_run_id = "004"

In [39]:
RAG_MEMORY_PATH = "memory/wikidyk_rag.parquet"
TRAINING_STUDENT_MEMORY_PATH = f"checkpoints/training_{training_run_id}"

AGENT_CONFIG = {
    "expensive": False,
    "provider": "anthropic",
    "cache": False
}

In [8]:
n_wikidyk = 5

# Prepare Memories: WikiDYK

### Data

In [40]:
wikidyk = pd.read_parquet("hf://datasets/YWZBrandon/wikidyk/data/test-00000-of-00001.parquet")
wikidyk_data = wikidyk[["fact", "eval"]].drop_duplicates().head(100)

/Users/henrikseng/miniforge3/envs/student/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [42]:
"""
wikidyk_data.head(1).to_dict()
{'fact': {0: 'Umar Zahir built both an island of trash and an island of hope'},
 'eval': {0: '{"reliability": {"prompt": "Who built both an island of trash and an island of hope?", "answer": ["Umar Zahir"]}, "generality": {"prompt": "What did Umar Zahir build alongside an island of hope?", "answer": ["an island of trash"]}, "paraphrase": {"prompt": "Which individual created both a trash island and an island of hope?", "answer": ["Umar Zahir"]}, "factual": {"prompt": "Umar Zahir built both an island of trash and an island of hope. Is this statement true or false?", "answer": ["true"]}, "counterfactual": {"prompt": "Umar Zahir built neither an island of trash nor an island of hope. Is this statement true or false?", "answer": ["false"]}}'}}
"""
# explode the eval column into separate columns
wikidyk_data = wikidyk_data.join(pd.json_normalize(wikidyk_data['eval']).add_prefix('eval_'))

In [43]:
columns = ['reliability.prompt',
 'reliability.answer',
 'generality.prompt',
 'generality.answer',
 'paraphrase.prompt',
 'paraphrase.answer',
 'factual.prompt',
 'factual.answer',
 'counterfactual.prompt',
 'counterfactual.answer',
 'portability.prompt',
 'portability.answer',
 'locality.prompt',
 'locality.answer']

In [44]:
eval = pd.json_normalize(wikidyk_data['eval'].apply(json.loads))

In [45]:
eval = eval[[c for c in eval.columns if c.endswith('.prompt') or c.endswith('.answer') and not "alternative" in c]]
eval

,reliability.prompt,reliability.answer,generality.prompt,generality.answer,paraphrase.prompt,paraphrase.answer,factual.prompt,factual.answer,counterfactual.prompt,counterfactual.answer,portability.prompt,portability.answer,locality.prompt,locality.answer
0,Who built both an island of trash and an islan...,[Umar Zahir],What did Umar Zahir build alongside an island ...,[an island of trash],Which individual created both a trash island a...,[Umar Zahir],Umar Zahir built both an island of trash and a...,[true],Umar Zahir built neither an island of trash no...,[false],NaN,NaN,NaN,NaN
1,What is the name of the song for which Kanye W...,[Gold Digger],From whose point of view did Kanye West origin...,[female],What is the title of the song where Kanye West...,[Gold Digger],"Kanye West originally wrote the chorus of "" Go...",[true],Kanye West originally wrote the chorus of 'Gol...,[false],I recently came across a story about an Americ...,[Gold Digger],"Which American artist, born in 1977, revolutio...",Kanye West
2,"Who, along with his colleagues, found in 2019 ...",[Paul Cosford],In what year did Paul Cosford and his colleagu...,[2019],Who discovered in 2019 with his team that lung...,[Paul Cosford],"in 2019, Paul Cosford and his colleagues found...",[true],"In 2019, Paul Cosford and his colleagues found...",[false],I recently read an article about a severe lung...,[Paul Cosford],What is a type of malignancy that originates i...,Lung cancer
3,What is the name of the video game developed i...,[Pyongyang Racer],In which country was the video game Pyongyang ...,[North Korea],Can you tell me the name of the video game cre...,[Pyongyang Racer],the video game Pyongyang Racer was developed i...,[true],The video game Pyongyang Racer was developed i...,[false],I recently heard about a British-founded trave...,[Pyongyang Racer],Which British-founded travel company based in ...,Koryo Tours
4,Which AI expert resigned her role as a CEO to ...,[Tess Posner],What role did Tess Posner resign from to conce...,[CEO],Who stepped down from being a CEO to pursue th...,[Tess Posner],AI expert Tess Posner resigned her role as a C...,[true],AI expert Tess Posner was promoted to a CEO ro...,[false],NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,What has been proposed as the official state c...,[Colby cheese],Which U.S. state has seen several attempts to ...,[Wisconsin],Which cheese has been suggested multiple times...,[Colby cheese],there have been several attempts to make Colby...,[true],There have been no attempts to make Colby chee...,[false],I recently learned about an Upper Midwest stat...,[Colby cheese],"Which ancient Sanskrit poet and playwright, ac...",Kalidasa
96,What is the viral online word game that was or...,[Wordle],Who was the viral online word game Wordle orig...,[the developer and his partner],Which viral online word game was initially inv...,[Wordle],the viral online word game Wordle (instance pi...,[true],The viral online word game Wordle was original...,[false],NaN,NaN,NaN,NaN
97,Whose home was demolished due to an Indonesian...,"[a former public works minister, Martinus Putu...",What type of project resulted in the demolitio...,[road expansion project],Who had their home demolished because of an In...,"[a former public works minister, Martinus Putu...",an Indonesian road expansion project demolishe...,[true],An Indonesian road expansion project preserved...,[false],NaN,NaN,NaN,NaN
98,Where was a Māori military settlement establis...,"[Māngere Bridge, New Zealand]",Who established a military settlement at Mānge...,[Māori],"In the 1840s, where was a Māori military base ...","[Māngere Bridge, New Zealand]","a Māori military settlement at Māngere Bridge,...",[true],"A Māori military settlement at Māngere Bridge,...",[false],I was reading about the indigenous people of m...,"[Māngere Bridge, New Zealand]",Which medieval Italian Dominican friar and the...,Thomas Aquinas


In [40]:
eval["counterfactual.answer"].apply(lambda x: x[0]).unique()

array(['false'], dtype=object)

In [24]:
#json.loads(wikidyk_data_sampled.iloc[0].to_dict()["eval"])["generality"]

In [5]:
#facts = list(wikidyk_data_sampled["fact"])
facts = list(wikidyk_data["fact"])[:100]

### Naive

In [30]:
memory_rag_naive = MemoryRAG() # populate manually
memory_rag_naive

In [31]:
for fact in facts:
    new_node = MemoryNodeRAG(input=fact)
    memory_rag_naive.add(new_node)

In [33]:
[len(node.embeddings) for node in memory_rag_naive.memory.values()][:5] # all 0

[0, 0, 0, 0, 0]

In [27]:
# import litellm
# litellm._turn_on_debug()

##### Set embeddings once by recalling anything

In [36]:
# memory_rag_naive.recall("What was named after Olena Stepaniv in Lviv?", sensitivity=0.1, thres=0.1)
nodes = memory_rag_naive.get_nodes() # set embeddings

In [38]:
{len(node.embeddings) for node in memory_rag_naive.memory.values()} # all 0# all 1

{1}

In [30]:
memory_rag_naive.save(RAG_MEMORY_PATH)

#### Verify that save / load works

In [31]:
m = MemoryRAG()
m.load(RAG_MEMORY_PATH)

In [32]:
for node in m.memory.values():
    print(len(node.embeddings)) # all 1

m.recall("What was named after Olena Stepaniv in Lviv?", sensitivity=0.1, thres=0.1)

1
1
1
1
1


{'d56df78e': 'postcards were made of Olena Stepaniv during the First World War, and in 1991 Lviv named a street after her',
 '7c9efd08': 'on the 100th anniversary of International Women\'s Day, Peninah Musyimi, from the slums of Nairobi, was given the "I am Powerful" award',
 '08223430': 'the Empire of Japan turned a Korean royal cemetery at what is now Hyochang Park into a golf course',
 '606689a1': 'Spanish mystic Marina de Escobar founded a convent but never joined one'}

In [33]:
memory_rag_agentic = MemoryRAG() # from StudentAgent

### Student

In [24]:
# Initialization

In [37]:

teaching_prompt = "You are an expert in collecting factual knowledge in your memory. Memorize the facts explicitly. (NO verification required)"
student_wiki = StudentAgent(**AGENT_CONFIG)
student_wiki.reset_system_prompt(teaching_prompt, append=True)
student_wiki.save(TRAINING_STUDENT_MEMORY_PATH)

In [38]:
# Teaching procedure

def train_memory(fact):
    student_wiki.load(TRAINING_STUDENT_MEMORY_PATH)
    student_wiki.reset_chat()
    p = f"Fact: {fact}"
    student_wiki.run(p, remove_tools=["ask memory"])
    student_wiki.reset_chat()
    student_wiki.save(TRAINING_STUDENT_MEMORY_PATH)

In [39]:
for j, fact in enumerate(facts):
    print(j)
    train_memory(fact)

0

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

1

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

2
3
4

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



In [27]:
STUDENT_MEMORY_PATH = f"checkpoints/memory_{training_run_id}"

student_wiki = StudentAgent(**AGENT_CONFIG)
student_wiki.load(STUDENT_MEMORY_PATH)
#student_wiki.save(STUDENT_MEMORY_PATH)
#student_wiki.reset_conversation()

##### Memory dimensions

In [36]:
student_wiki = StudentAgent(**AGENT_CONFIG)
student_wiki.load(f"checkpoints/memory_run__100")
print("Memory size: ", student_wiki.memory_size())

Memory size:  177


In [35]:
def mean(values):
    return sum(values) / len(values) if values else 0

print("Average number of keys per memory entry: ", mean([len(list(k.keys)) for k in student_wiki.memory_agent.memory.memory.values()]))

Average number of keys per memory entry:  5.937853107344632


In [56]:
#student_wiki.memory_agent.memory.render()
#student_wiki.render_conversation()

# Training: Student vs Baselines

- StudentAgent:     all data --> memory --> ask

- pretraining ~     LLM
- answerable  ~     LLM + fact
- AgenticRAG  ~     LLM + RAG @ frozen memory
- NaiveRAG    ~     LLM + RAG @ data

In [ ]:
#baseline_naive = NaiveRAGAgent(memory_rag_naive)
#baseline_agentic = AgenticRAGAgent(memory_rag_agentic)

### Setup Experiments

In [41]:
from benchmark import Experiment, configure_logging, run_experiments_parallel

In [1]:
RAG_MEMORY_PATH = "memory/wikidyk_rag.parquet"
STUDENT_MEMORY_PATH = "checkpoints/training_003"

AGENT_CONFIG = {
    "expensive": False,
    "provider": "anthropic",
    "cache": False,
}
AGENT_IDS = [
    "baseline_naive",
    "baseline_agentic",
    "student",
    "baseline_pretraining",
    "baseline_answerable",
]



In [9]:
run_id = "test_0000"
base = {
    "run_id": run_id,
    "fact_id": 4,
    "fact": "Paris is the biggest city of France and its capital.",
    "question_type": "reliability",
    "question": "What is the capital of France?",
    "correct_answer": "Paris",
}

experiments = []
for agent in AGENT_IDS:
    cfg = dict(base)
    cfg["agent_id"] = agent
    experiments.append(cfg)

summary = run_experiments_parallel(
    experiments,
    outfile=f"results/results__{run_id}.jsonl",
    max_workers=1,           
    per_task_timeout=None,   # or e.g. 120 for 2 min per task
)

print("Summary:", {k: v for k, v in summary.items() if k != "results_sample" and k != "failures"})

Memory converted to RAG memory
Memory converted to RAG memory
Summary: {'total': 5, 'completed': 5, 'failed': 0, 'outfile': 'results/results__test_0000.jsonl'}


In [ ]:
run_id = "test_001"

experiments = []
for i, row in wikidyk_data[:2].iterrows():
    fact_id = str(i)
    fact = row["fact"]
    evals = json.loads(row["eval"]) if isinstance(row["eval"], str) else row["eval"]
    for question_type, qa in evals.items():
        question = qa["prompt"]
        correct_answer = qa["answer"]
        for agent_id in ["baseline_naive", "baseline_agentic"]:
            
            exp = {
                "run_id": run_id,
                "agent_id": agent_id,
                "fact_id": str(i),
                "fact": fact,
                "question_type": question_type,
                "question": question,
                "correct_answer": correct_answer,
            }
            experiments.append(exp)


In [66]:
summary = run_experiments_parallel(
    experiments,
    outfile=f"results/results__{run_id}.jsonl",
    max_workers=1,           
    per_task_timeout=None,   # or e.g. 120 for 2 min per task
)

print("Summary:", {k: v for k, v in summary.items() if k != "results_sample" and k != "failures"})

Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Memory converted to RAG memory
Summary: {'total': 14, 'completed': 14, 'failed': 0, 'outfile': 'results/results__test_002.jsonl'}


## Test baseline agents

In [1]:
from student.agent import *
from student.agent.memory_rag import MemoryRAG, MemoryNodeRAG

In [2]:
RAG_MEMORY_PATH = "memory/wikidyk_rag__run__100.parquet"

In [5]:
m = MemoryRAG()
m.load(RAG_MEMORY_PATH)
m

In [ ]:
naive = NaiveRAGAgent(memory=m)
naive.memory_size()

In [9]:
naive = AgenticRAGAgent(memory_path=RAG_MEMORY_PATH)
naive.memory_size()

100

In [10]:
naive.run("Who built both an island of trash and an island of hope?")

'Umar Zahir built both an island of trash and an island of hope.'

In [43]:
RESULTS_PATH = "results/results__benchmark__run__100.jsonl"

rows = load_jsonl(RESULTS_PATH)

# remove rows that have "result": {"error" : {litellm.litellm.InternalServerError*}...}
rows = [r for r in rows if not r.get("result", {}).get("error", {}).startswith("litellm.litellm.InternalServerError")]

AttributeError: 'str' object has no attribute 'get'

In [62]:
# rows = [r for r in rows if not r.get("result", {}).get("error", {}).startswith("litellm.litellm.InternalServerError")]

rows_removed = []
for r in rows:
    res = r.get("result", {})
    if type(res) is dict:    
        error = res.get("error", {})
        if error.startswith("litellm.InternalServerError"):
            continue
    rows_removed.append(r)

print("Removed ", len(rows) - len(rows_removed), " rows")

Removed  17  rows


In [57]:
len(rows[717:720])

3

# Evaluation

#### Setup

In [ ]:
import argparse
import json
import logging
import os
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from metrics import metrics

def configure_logging(level: int = logging.INFO):
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )


def _safe_extract_output(result: Any) -> Optional[str]:
    if result is None:
        return None
    if isinstance(result, str):
        return result.strip()
    if isinstance(result, dict):
        # common patterns
        if "error" in result and result["error"]:
            return None
        for key in ("text", "output", "answer", "response", "content"):
            if key in result and isinstance(result[key], str):
                return result[key].strip()
        # last resort: stringify
        return json.dumps(result)
    # fallback
    return str(result)


def load_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            ln = line.strip()
            if not ln:
                continue
            try:
                rows.append(json.loads(ln))
            except Exception as e:
                logging.warning(f"Skipping bad JSON line {i}: {e}")
                continue
    return rows


def compute_metrics_df(rows: List[Dict[str, Any]]) -> pd.DataFrame:
    records = []
    for r in rows:
        agent_id = r.get("agent_id")
        qtype = r.get("question_type")
        correct = r.get("correct_answer")
        result = r.get("result")
        output = _safe_extract_output(result)

        computed = {"match": None, "f1": None}
        if output is not None and correct is not None:
            try:
                m = metrics(correct[0], output)  # user-provided
                computed["match"] = 1 if bool(m.get("match")) else 0
                f1val = m.get("f1")
                computed["f1"] = float(f1val) if f1val is not None else None
            except Exception as e:
                logging.warning(f"metrics() failed for agent_id={agent_id}, qtype={qtype}: {e}")

        base = {
            "agent_id": agent_id,
            "run_id": r.get("run_id"),
            "fact_id": r.get("fact_id"),
            "question_type": qtype,
            "question": r.get("question"),
            "correct_answer": correct,
            "model_output": output,
        }
        base.update(computed)
        records.append(base)

    df = pd.DataFrame.from_records(records)
    return df


def aggregate_by_agent_qtype(df: pd.DataFrame) -> pd.DataFrame:
    grp = (
        df.dropna(subset=["match", "f1"])
          .groupby(["agent_id", "question_type"], as_index=False)
          .agg(match_acc=("match", "mean"), f1=("f1", "mean"))
    )
    return grp


def _ordered_qtypes(unique_qtypes: List[str]) -> List[str]:
    # Preferred order; fall back to whatever exists
    preferred = ["Reliability", "Generality", "Paraphrase", "Portability", "Locality"]
    upper_map = {q.upper(): q for q in unique_qtypes}
    ordered = [upper_map[q.upper()] for q in preferred if q.upper() in upper_map]
    # include any remaining qtypes not in preferred
    for q in unique_qtypes:
        if q not in ordered:
            ordered.append(q)
    return ordered


def _grouped_bar_plot(pivot_df: pd.DataFrame, title: str, ylabel: str, out_path: str):
    # pivot_df: index=agent_id, columns=question_type, values = metric (%)
    agents = list(pivot_df.index)
    qtypes = list(pivot_df.columns)

    x = np.arange(len(agents))
    n = len(qtypes)
    width = 0.8 / max(n, 1)  # keep total width reasonable

    plt.figure(figsize=(10, 4))
    for i, q in enumerate(qtypes):
        vals = pivot_df[q].values
        plt.bar(x + i * width - (n - 1) * width / 2.0, vals, width=width, label=q)

    plt.xticks(x, agents, rotation=20, ha="right")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(title="Question Type")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def make_plots(agg: pd.DataFrame, outdir: str):
    os.makedirs(outdir, exist_ok=True)

    # Order columns (qtypes) consistently
    q_order = _ordered_qtypes(sorted(agg["question_type"].dropna().unique().tolist()))

    # Match accuracy (%)
    pivot_match = (
        agg.assign(match_pct=agg["match_acc"] * 100.0)
           .pivot_table(index="agent_id", columns="question_type", values="match_pct", fill_value=0.0)
           .reindex(columns=[q for q in q_order if q in agg["question_type"].unique()])
           .sort_index()
    )
    _grouped_bar_plot(
        pivot_match,
        title="Match Accuracy by Agent (higher is better)",
        ylabel="Match Accuracy (%)",
        out_path=os.path.join(outdir, "match_accuracy_by_agent.png"),
    )

    # F1 (%)
    pivot_f1 = (
        agg.assign(f1_pct=agg["f1"])
           .pivot_table(index="agent_id", columns="question_type", values="f1_pct", fill_value=0.0)
           .reindex(columns=[q for q in q_order if q in agg["question_type"].unique()])
           .sort_index()
    )
    _grouped_bar_plot(
        pivot_f1,
        title="F1 by Agent (higher is better)",
        ylabel="F1 (%)",
        out_path=os.path.join(outdir, "f1_by_agent.png"),
    )

    # also save CSVs for convenience
    pivot_match.to_csv(os.path.join(outdir, "match_accuracy_by_agent.csv"))
    pivot_f1.to_csv(os.path.join(outdir, "f1_by_agent.csv"))


def _parse_agent_to_model_obj(agent_id: str, amap: Optional[Dict[str, Dict[str, str]]] = None):
    if amap and agent_id in amap:
        md = amap[agent_id]
        return md.get("Model", agent_id), md.get("Obj", "")
    # default: leave as-is
    return agent_id, ""


def build_latex_table_df(agg: pd.DataFrame, agent_map: Optional[Dict[str, Dict[str, str]]] = None) -> pd.DataFrame:
    # Prepare percentages
    df = agg.copy()
    df["Match"] = df["match_acc"] * 100.0
    df["F1"] = df["f1"] * 100.0

    # Identify question types in preferred order
    q_order = _ordered_qtypes(sorted(df["question_type"].dropna().unique().tolist()))

    # Build a wide table per agent with two columns (Match, F1) for each qtype
    # First, reshape so we can pivot both metrics
    melted = df.melt(
        id_vars=["agent_id", "question_type"],
        value_vars=["Match", "F1"],
        var_name="metric",
        value_name="value",
    )

    wide = (
        melted.pivot_table(
            index="agent_id",
            columns=["question_type", "metric"],
            values="value",
            aggfunc="mean",
        )
        .reindex(columns=pd.MultiIndex.from_product([q_order, ["Match", "F1"]]))
        .sort_index()
    )

    # Convert to a regular DataFrame with Model/Obj. up front
    model_obj_rows = [ _parse_agent_to_model_obj(aid, agent_map) for aid in wide.index ]
    model_col = [m for (m, _) in model_obj_rows]
    obj_col = [o for (_, o) in model_obj_rows]

    # Format as 2 decimal strings
    formatted = wide.applymap(lambda x: f"{x:.2f}" if pd.notnull(x) else "")

    # Build final frame
    formatted.insert(0, ("_", "Obj."), obj_col)     # temporary prefix for stable order
    formatted.insert(0, ("_", "Model"), model_col)
    formatted.columns = pd.MultiIndex.from_tuples(formatted.columns)

    # Sort columns: Model, Obj., then qtypes x (Match,F1)
    cols = [("_", "Model"), ("_", "Obj.")] + [(q, sub) for q in q_order for sub in ("Match", "F1")]
    formatted = formatted.reindex(columns=pd.MultiIndex.from_tuples(cols))

    # Remove the helper top-level "_" for the first two columns by flattening afterwards
    formatted.columns = pd.MultiIndex.from_tuples(
        [("Model", "") if c == ("_", "Model") else
         ("Obj.", "") if c == ("_", "Obj.") else c for c in formatted.columns]
    )

    # Reset index to turn agent_id into rows (will be replaced by Model/Obj.)
    formatted = formatted.reset_index(drop=True)

    return formatted


def save_latex_table(df_latex: pd.DataFrame, path: str):
    # Use to_latex with MultiIndex columns, no escaping (we aren't adding LaTeX chars here)
    with open(path, "w", encoding="utf-8") as f:
        f.write(df_latex.to_latex(index=False, escape=False, multicolumn=True, multicolumn_format='c'))

#### Test run

In [8]:
RESULTS_PATH = "results/results__test_run_001.jsonl"
OUTDIR = "output/figures/test_001"
LATEX_TABLE_PATH = "output/tables/"

In [16]:
configure_logging()

rows = load_jsonl(RESULTS_PATH)

df = compute_metrics_df(rows)
agg = aggregate_by_agent_qtype(df)

In [17]:
os.makedirs(OUTDIR, exist_ok=True)
df.to_csv(os.path.join(OUTDIR, "per_example_with_metrics.csv"), index=False)
agg.to_csv(os.path.join(OUTDIR, "agg_by_agent_qtype.csv"), index=False)

make_plots(agg, OUTDIR)

In [20]:
# LaTeX export
agent_map = None

latex_df = build_latex_table_df(agg, agent_map=agent_map)
latex_df.to_csv(os.path.join(LATEX_TABLE_PATH, "latex_table_preview.csv"), index=False)

os.makedirs(os.path.dirname(LATEX_TABLE_PATH), exist_ok=True)
save_latex_table(latex_df, os.path.join(LATEX_TABLE_PATH, "latex_table.tex"))

logging.info(f"Analysis complete. Outputs in: {OUTDIR}")
if LATEX_TABLE_PATH:
    logging.info(f"LaTeX table saved to: {LATEX_TABLE_PATH}")


/var/folders/6p/s0rnd_zn3zvbjzlxtjryt_t80000gn/T/ipykernel_1865/4070005341.py:265: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  formatted = wide.applymap(lambda x: f"{x:.2f}" if pd.notnull(x) else "")
2025-08-22 00:32:42,374 | INFO | Analysis complete. Outputs in: output/figures/test_001
2025-08-22 00:32:42,374 | INFO | LaTeX table saved to: output/tables/


In [ ]:
def main():
    configure_logging()

    rows = load_jsonl(RESULTS_PATH)
    if not rows:
        logging.error(f"No rows loaded from {RESULTS_PATH}. Exiting.")
        return

    df = compute_metrics_df(rows)
    agg = aggregate_by_agent_qtype(df)

    os.makedirs(OUTDIR, exist_ok=True)
    # Save raw with metrics
    df.to_csv(os.path.join(OUTDIR, "per_example_with_metrics.csv"), index=False)
    # Save aggregation
    agg.to_csv(os.path.join(OUTDIR, "agg_by_agent_qtype.csv"), index=False)

    # Plots
    make_plots(agg, OUTDIR)

    # LaTeX export
    agent_map = None

    latex_df = build_latex_table_df(agg, agent_map=agent_map)
    latex_df.to_csv(os.path.join(OUTDIR, "latex_table_preview.csv"), index=False)

    if LATEX_TABLE_PATH:
        os.makedirs(os.path.dirname(LATEX_TABLE_PATH) or ".", exist_ok=True)
        save_latex_table(latex_df, LATEX_TABLE_PATH)

    logging.info(f"Analysis complete. Outputs in: {OUTDIR}")
    if LATEX_TABLE_PATH:
        logging.info(f"LaTeX table saved to: {LATEX_TABLE_PATH}")


if __name__ == "__main__":
    main()


In [21]:
RESULTS_PATH = "results/results__benchmark__run__100.jsonl"
OUTDIR = "output/figures/test__104"
LATEX_TABLE_PATH = "output/tables/"


rows = load_jsonl(RESULTS_PATH)
rows_corrected = load_jsonl("results/results__benchmark__run__101.jsonl")
rows_rag = load_jsonl("results/results__benchmark__run__103.jsonl")

# drop all rows with agent_id == "baseline_answerable" and combine rows_corrected into rows
rows = [r for r in rows if r.get("agent_id") not in ["baseline_answerable","baseline_naive", "baseline_agentic"]]
rows.extend(rows_corrected)
rows.extend(rows_rag)


df = compute_metrics_df(rows)
agg = aggregate_by_agent_qtype(df)
# remove the f1 scores for factuals and counterfactuals by substituting agg["f1"]=0
agg.loc[(agg["question_type"].isin(["factual", "counterfactual"])) & (agg["f1"] > 0), "f1"] = 0

In [22]:
agg

,agent_id,question_type,match_acc,f1
0,baseline_agentic,counterfactual,0.850000,0.000000
1,baseline_agentic,factual,0.633663,0.000000
2,baseline_agentic,generality,0.544554,53.356907
3,baseline_agentic,locality,0.000000,0.000000
4,baseline_agentic,paraphrase,0.712871,62.211221
5,baseline_agentic,portability,0.681159,56.487233
6,baseline_agentic,reliability,0.722772,67.095710
7,baseline_answerable,counterfactual,0.960000,0.000000
8,baseline_answerable,factual,0.870000,0.000000
9,baseline_answerable,generality,0.690000,87.944218


# TODO

1. Check prompts of agents and optimize
2. Select subset
3. Run for larger set

In [ ]:
from mllm import Chat
c = Chat()

In [ ]:
c += "Respond with a random letter"
res = c.complete(expensive=False)
c += "Respond with two random letters"
res = c.complete(expensive=False)
input_tokens = c.additional_res["prompt_tokens"]
output_tokens = c.additional_res["completion_tokens"]
c.additional_res

In [1]:
from student.agent import Agent, StudentAgent

In [4]:
x = StudentAgent(expensive=False)
m = x.get_memory_agent()
#m.chat_config(cache=True, expensive=False)
m.expensive

False

In [13]:
print(x.token_counter, m.token_counter)

[{'input_tokens': 701, 'output_tokens': 27}, {'input_tokens': 12, 'output_tokens': 1}] [{'input_tokens': 167, 'output_tokens': 1}, {'input_tokens': 569, 'output_tokens': 74}, {'input_tokens': 726, 'output_tokens': 60}, {'input_tokens': 869, 'output_tokens': 59}, {'input_tokens': 1011, 'output_tokens': 57}, {'input_tokens': 1151, 'output_tokens': 38}, {'input_tokens': 1392, 'output_tokens': 150}, {'input_tokens': 1808, 'output_tokens': 69}, {'input_tokens': 1134, 'output_tokens': 54}]


In [7]:
x.run("What is the capital of France?")

'The capital of France is Paris.'

In [10]:
x.single_run("Respond with a random letter")

'Q'

In [12]:
x.memory_agent.learn("My name is henrik. Memorize this")

'Learning successful! No clarifications are needed at this time.'

In [20]:
x.reset_token_count()
# {'input_tokens': 9540, 'output_tokens': 590}

{'input_tokens': 9540, 'output_tokens': 590}

In [19]:
def cost_estimation(tokens, prizes_per_mil):
    total_cost = 0
    for key, value in tokens.items():
        total_cost += value * prizes_per_mil.get(key, 0) / 1000000
    return total_cost

cost_estimation(x.sum_token_count(), {"input_tokens":0.8,"output_tokens":4})

0.009992000000000001

In [3]:
student = StudentAgent(**AGENT_CONFIG)
student.load(STUDENT_MEMORY_PATH)
student.setup_quiz()

In [4]:
student.run("What is the capital of france?")

'Paris'

In [ ]:
from student.agent.tools.tools import Tool
class RespondHumanTool(Tool):
    def __init__(self):
        super().__init__("respond_human", "Provide a response or update to the human user")

    def run(self, message: str) -> str:
        """Provide a response or update to the human user."""
        return f"Response to human: {message}"



In [1]:
from student.agent.agent_baselines import *

In [2]:
b = BaselineAgent(expensive=False, cache=False)

In [3]:
b.run_answerable(**{
    'question': 'Umar Zahir built both an island of trash and an island of hope. Is this statement true or false?',
    'context': 'Umar Zahir built both an island of trash and an island of hope.'
})

You are tasked with a knowledge test. 
        You have to answer the <question> as precise and short as possible.

        Inputs:
        <question>
        Umar Zahir built both an island of trash and an island of hope. Is this statement true or false?
        </question>
        <context>
        Umar Zahir built both an island of trash and an island of hope.
        </context>

        Instructions:
        - Output the precise answer for the <question>

        Example input: 
        <question>
        What is the capital of France?
        </question>
        <context>
        Paris is the capital of France.
        </context>

        Example outputs (IMPORTANT only precise + short answers. NO SENTENCES):
        Paris
        


'True'

In [4]:
from benchmark import *

In [5]:
n_wikidyk = 2
start = 0
end = 2

wikidyk = pd.read_parquet("hf://datasets/YWZBrandon/wikidyk/data/test-00000-of-00001.parquet")
wikidyk_data = wikidyk[["fact", "eval"]].drop_duplicates()

run_id = "test__run__100"

/Users/henrikseng/miniforge3/envs/student/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
experiments = []
for i, row in wikidyk_data[:n_wikidyk].iterrows():
    if i not in range(start, end):
        continue
    
    fact_id = str(i)
    fact = row["fact"]
    evals = json.loads(row["eval"]) if isinstance(row["eval"], str) else row["eval"]
    for question_type, qa in evals.items():
        question = qa["prompt"]
        correct_answer = qa["answer"]
        for agent_id in ["baseline_answerable"]:
            
            exp = {
                "run_id": run_id,
                "agent_id": agent_id,
                "fact_id": str(i),
                "fact": fact,
                "question_type": question_type,
                "question": question,
                "correct_answer": correct_answer,
            }
            experiments.append(exp)




In [7]:
run_experiments_parallel([experiments[3]], max_workers=1, outfile="test.jsonl")

You are tasked with a knowledge test. 
        You have to answer the <question> as precise and short as possible.

        Inputs:
        <question>
        Umar Zahir built both an island of trash and an island of hope. Is this statement true or false?
        </question>
        <context>
        Umar Zahir built both an island of trash and an island of hope
        </context>

        Instructions:
        - Output the precise answer for the <question>

        Example input: 
        <question>
        What is the capital of France?
        </question>
        <context>
        Paris is the capital of France.
        </context>

        Example outputs (IMPORTANT only precise + short answers. NO SENTENCES):
        Paris
        


{'total': 1,
 'completed': 1,
 'failed': 0,
 'outfile': 'test.jsonl',
 'results_sample': [{'agent_id': 'baseline_answerable',
   'run_id': 'test__run__100',
   'fact_id': '0',
   'fact': 'Umar Zahir built both an island of trash and an island of hope',
   'question_type': 'factual',
   'question': 'Umar Zahir built both an island of trash and an island of hope. Is this statement true or false?',
   'correct_answer': ['true'],
   'result': 'False',
   '_meta': {'experiment_id': 'test__run__100__0__factual__baseline_answerable',
    'started_at': '2025-08-23T20:35:02.116246+00:00',
    'finished_at': '2025-08-23T20:35:02.952729+00:00',
    'duration_sec': 0.8365,
    'agent_config': {'expensive': False,
     'provider': 'anthropic',
     'cache': False},
    'write_file': 'test.jsonl',
    'token_count': {'input_tokens': 166, 'output_tokens': 4}}}],
 'failures': []}

In [8]:
exp = Experiment(experiments[3])

In [9]:
def run_agent(self, agent):
    if agent is None:
        return {"error": "Agent init failed"}

    prompt = f"Question: {self.question}\nAnswer (no sentence, just precise keyword): "

    try:
        if self.agent_id in ["baseline_naive", "student", "baseline_agentic"]:
            return agent.run(prompt)

        elif self.agent_id == "baseline_answerable":
            return agent.run_answerable(context=self.fact, question=self.question)

        elif self.agent_id == "baseline_pretraining":
            return agent.run_pretraining(question=self.question)

        else:
            return {"error": f"Invalid agent id at runtime: {self.agent_id}"}

    except Exception as e:
        logger.exception(f"[{self.identifier()}] Agent run failed: {e}")
        return {"error": str(e)}

In [10]:
agent = exp.get_agent()

In [11]:
run_agent(exp, agent)

You are tasked with a knowledge test. 
        You have to answer the <question> as precise and short as possible.

        Inputs:
        <question>
        Umar Zahir built both an island of trash and an island of hope. Is this statement true or false?
        </question>
        <context>
        Umar Zahir built both an island of trash and an island of hope
        </context>

        Instructions:
        - Output the precise answer for the <question>

        Example input: 
        <question>
        What is the capital of France?
        </question>
        <context>
        Paris is the capital of France.
        </context>

        Example outputs (IMPORTANT only precise + short answers. NO SENTENCES):
        Paris
        


'False'

In [ ]:
from mllm import Chat
c = Chat(dedent=True)
c += """You are tasked with a knowledge test. 
        You have to answer the <question> as precise and short as possible.

        Inputs:
        <question>
        Umar Zahir built both an island of trash and an island of hope. Is this statement true or false?
        </question>
        <context>
        Umar Zahir built both an island of trash and an island of hope
        </context>

        Instructions:
        - Output the precise answer for the <question>
        - ONLY precise answers. NO SENTENCES
"""

In [20]:
c.complete(expensive=False)

'True'